# 02 · La convolución
*Un kernel chico que se desliza: suma de productos, padding y stride*

Notebook del episodio 2 de **Visión: de convoluciones a ViT** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/vision/la-convolucion)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** kernel [−1 0 1] a mano sobre un borde 5×5 (1|8): salida 3×3 con columnas 0, 21, 21 → Sobel sobre la flor (gris = 0,299R + 0,587G + 0,114B): salida (425, 638), 2,4 % de píxeles con borde > 1; ventana real g_x = 1,76; heatmaps de g_x con signo y |∇g| en la flor achicada 26 × 40 → F.conv2d = loop (dif. 1,4e-6) → o = ⌊(n + 2p − k)/s⌋ + 1: 30, 32, 16, 15 y padding='same' (Dumoulin & Visin, relación 6) → F.conv2d es correlación cruzada: igual a correlate2d, difiere 9,27 de convolve2d, igual con el kernel girado

**Cómo correrlo:** con el entorno de `~/Movies/serie-vision/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "vision"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Visión, de las convoluciones a los transformers. Episodio dos: la convolución. En el episodio anterior vimos que una imagen es un tensor de números y que una red densa no sabe qué píxel está al lado de cuál. Hoy vemos la operación que sí lo sabe, con números a mano, con una foto real, y en PyTorch.*

### Qué vamos a ver

- Un kernel chico que se desliza por la imagen
- En cada posición: una suma de productos
- Padding y stride deciden el tamaño de salida
- PyTorch hace correlación cruzada, y da igual

Cuatro ideas. Un kernel, que es una matriz chiquita de pesos, de tres por tres por ejemplo, que se desliza por toda la imagen. En cada posición se multiplica el kernel por el pedazo de imagen que tiene abajo, número a número, y se suma todo: sale un solo número. Dos perillas, el padding y el stride, deciden de qué tamaño queda el resultado, y hay una fórmula exacta. Y un detalle de nombres: lo que PyTorch llama convolución, en matemática se llama correlación cruzada.

## Parte 1 · Suma de productos local

Arranquemos con una imagen tan chica que entra entera en pantalla, para hacer la cuenta a mano.

### Una imagen de 5 × 5

Esta es la imagen: veinticinco píxeles de un solo canal, en escala de grises. Las tres columnas de la izquierda valen uno, oscuras, y las dos de la derecha valen ocho, claras. Entre la columna dos y la tres hay un borde vertical: el brillo salta de golpe. La pregunta del episodio es cómo hace una red para encontrar ese borde sin que nadie le diga dónde está.

### La convolución con dos loops

Así se escribe con dos loops de Python. x es la imagen de cinco por cinco. w es el kernel de tres por tres: menos uno en la columna izquierda, cero en el medio, más uno en la derecha. O sea: el brillo de la derecha menos el de la izquierda. La salida tiene tres por tres, porque un kernel de tres entra en tres posiciones de cinco. Para cada posición i j recortamos la ventana de tres por tres que empieza ahí. Y multiplicamos ventana por kernel elemento a elemento, y sumamos. Mirá el resultado: la primera columna da cero, donde todo es uniforme, y las otras dos dan veintiuno, justo donde la ventana pisa el borde.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep02_a_mano.py
__file__ = 'snippets/ep02_a_mano.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import torch
from _cifar import ruta_lab

In [3]:
x = torch.tensor([[1., 1., 1., 8., 8.]] * 5)   # 5 filas: oscuro | claro
w = torch.tensor([[-1., 0., 1.]] * 3)          # kernel: derecha − izquierda
k = 3
salida = x.shape[0] - k + 1                # 5 − 3 + 1 = 3
y = torch.zeros(salida, salida)
for i in range(salida):
    for j in range(salida):
        ventana = x[i:i + k, j:j + k]      # parche de 3 × 3
        y[i, j] = (ventana * w).sum()      # suma de productos
print(y)

tensor([[ 0., 21., 21.],
        [ 0., 21., 21.],
        [ 0., 21., 21.]])


In [4]:
# Verificación extra (corrió junto con el video)
prod = (x[0:3, 1:4] * w).flatten().tolist()
json.dump({"x": x.tolist(), "y": y.tolist(), "prod": prod}, open(ruta_lab("datos", "figs", "ep02_a_mano.json"), "w"))

### La cuenta de una celda

$$
y_{ij} = \sum_{a=0}^{2}\sum_{b=0}^{2} x_{i+a,\,j+b}\; w_{ab}
$$
<small>ventana que empieza en (i, j), por el kernel, y sumar</small>

$$
x \in \mathbb{R}^{5\times 5},\;\; w \in \mathbb{R}^{3\times 3} \;\Rightarrow\; y \in \mathbb{R}^{3\times 3}
$$
<small>5 − 3 + 1 = 3 posiciones por lado</small>

En símbolos: x es la imagen, una matriz, y x sub i más a, j más b es un píxel. w sub a b es el peso del kernel en la fila a, columna b; a y b van de cero a dos. Entonces y sub i j, la salida en la celda i j, es la suma doble de píxel por peso sobre la ventana de tres por tres. En el dibujo, los nueve productos de la celda cero, uno: la columna izquierda aporta menos uno tres veces, la del medio cero, y la derecha ocho tres veces. Total: veintiuno. Y las formas: entra una matriz de cinco por cinco, el kernel es de tres por tres, y sale una de tres por tres, porque hay cinco menos tres más una posiciones por lado.

## Parte 2 · Bordes en una foto real

Con números inventados todo sale lindo. Probemos con una foto de verdad, de cuatrocientos veintisiete por seiscientos cuarenta píxeles.

### La foto, en escala de grises

Es una de las dos fotos de ejemplo que trae scikit-learn: una flor. Acá la ves en escala de grises y achicada dieciséis veces, promediando bloques de dieciséis por dieciséis, así cada píxel es un punto que se distingue. El centro de la flor es la mancha brillante; alrededor, hojas oscuras. Vamos a buscarle los bordes con un kernel clásico de procesamiento de imágenes, el de Sobel, que nadie entrenó: lo diseñó una persona.

### El filtro de Sobel, con F.conv2d

El código. Primero pasamos la foto de tres canales, rojo, verde y azul, a uno solo: el brillo g es una suma pesada de los tres, con los pesos clásicos de la televisión. El kernel de Sobel en x es el mismo que usamos a mano, pero la fila del medio pesa doble. Su transpuesta, filas por columnas, S sub y, mira bordes horizontales. F punto conv2d hace el deslizamiento; la estrellita es el símbolo de esa operación. F punto conv2d quiere cuatro dimensiones, batch, canal, alto y ancho: por eso los None. La fuerza del borde combina las dos direcciones con Pitágoras: raíz de g x al cuadrado más g y al cuadrado. Nabla g nombra ese par. La salida pierde un píxel por lado, y solo el dos coma cuatro por ciento de los píxeles tiene un borde fuerte.

**Cada línea, su término:**

- `gris = foto @` → $g = 0{,}299\,R + 0{,}587\,G + 0{,}114\,B$ — tres canales → uno: el brillo
- `sobel_x = torch.tensor` → $S_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}$ — derecha − izquierda, el centro pesa doble
- `gx = F.conv2d` → $g_x = g \star S_x$ — $\star$: deslizar y sumar productos
- `borde = (gx**2` → $\lvert\nabla g\rvert = \sqrt{g_x^2 + g_y^2}$ — cuánto cambia el brillo, en cualquier dirección

In [5]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep02_sobel.py
__file__ = 'snippets/ep02_sobel.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np
import torch
import torch.nn.functional as F
from _cifar import ruta_lab

FOTOS = ruta_lab("datos", "fotos.npz")

In [6]:
foto = torch.from_numpy(np.load(FOTOS)["flor"]).float() / 255
gris = foto @ torch.tensor([0.299, 0.587, 0.114])  # (H, W)
sobel_x = torch.tensor([[-1., 0., 1.],
                        [-2., 0., 2.],
                        [-1., 0., 1.]])
sobel_y = sobel_x.T                                  # bordes horizontales
x = gris[None, None]       # (batch, canal, alto, ancho)
gx = F.conv2d(x, sobel_x[None, None])
gy = F.conv2d(x, sobel_y[None, None])
borde = (gx**2 + gy**2).sqrt()
print("foto ", tuple(foto.shape), " gris", tuple(gris.shape))
print("borde", tuple(borde.shape[-2:]))
print(f"píxeles con borde fuerte (> 1): {(borde > 1).float().mean():.1%}")

foto  (427, 640, 3)  gris (427, 640)
borde (425, 638)
píxeles con borde fuerte (> 1): 2.4%


In [7]:
# Verificación extra (corrió junto con el video)
# Figuras (no se muestran): la misma cuenta sobre la foto achicada 16 veces (promedio de bloques de
# 16 × 16 píxeles), para que en pantalla se vea cada píxel. padding=1 para conservar el tamaño 26 × 40.
chica = F.avg_pool2d(x, 16)
gxc = F.conv2d(chica, sobel_x[None, None], padding=1)[0, 0]
gyc = F.conv2d(chica, sobel_y[None, None], padding=1)[0, 0]
bc = (gxc**2 + gyc**2).sqrt()
r = lambda t: [[round(float(v), 3) for v in fila] for fila in t]
# ventana real de 3 × 3 sobre el borde izquierdo de la flor (en la foto chica), para la figura de productos
fila_c, col_c = 13, int(gxc[13].argmax())
ventana = chica[0, 0, fila_c - 1:fila_c + 2, col_c - 1:col_c + 2]
json.dump({"gris": r(chica[0, 0]), "gx": r(gxc), "borde": r(bc),
           "ventana": r(ventana), "prod": [round(float(v), 3) for v in (ventana * sobel_x).flatten()],
           "gx_centro": round(float(gxc[fila_c, col_c]), 3), "pos": [fila_c, col_c]},
          open(ruta_lab("datos", "figs", "ep02_sobel.json"), "w"))

### Una ventana de la foto, número por número

$$
g_x(13, 10) = \sum_{a,b} g_{\text{ventana}} \odot S_x
$$
<small>$\odot$: producto elemento a elemento</small>

$$
g_x > 0:\ \text{sube el brillo hacia la derecha}
$$
<small>el signo dice para qué lado</small>

Una ventana real, en la foto achicada: fila trece, columna diez, el borde izquierdo de la flor. El círculo con punto es el producto elemento a elemento: cada píxel por el peso que tiene encima. En el dibujo, los nueve productos. La columna izquierda, oscura, resta poco; la derecha, más clara, suma bastante, y el centro pesa doble. Total: uno coma setenta y seis. El signo importa: positivo quiere decir que a la derecha hay más brillo que a la izquierda. Del otro lado de la flor va a dar negativo.

### Bordes verticales: azul sube, rojo baja

Y esta es la imagen entera de g sub x. Mirá la flor: el borde izquierdo sale azul, porque entrando a la flor el brillo sube, y el borde derecho sale rojo, porque saliendo baja. Arriba y abajo de la flor casi no hay nada: este kernel solo ve bordes verticales. El mismo kernel de nueve números, aplicado en todas las posiciones, encontró el contorno sin saber qué es una flor.

### La fuerza del borde

Juntando las dos direcciones con la raíz, queda la fuerza del borde sin signo. Ahora aparece el anillo completo: el contorno de la flor, también arriba y abajo. Kernels así, que detectan bordes en distintas orientaciones, son el tipo de cosa que suelen aprender solas las primeras capas de una red convolucional.

### F.conv2d hace lo mismo que el loop

Antes de seguir, confirmemos que F punto conv2d no hace nada mágico. Una imagen al azar de ocho por ocho, con las cuatro dimensiones: batch, canales, alto y ancho. Un kernel al azar, también con cuatro: canales de salida, canales de entrada, alto y ancho del kernel. El loop de antes contra F punto conv2d. Y la capa que se usa al armar redes, nn punto Conv2d, que guarda ese kernel como parámetro entrenable, más un bias, un número que se suma al final. La diferencia máxima es de una millonésima: errores de redondeo. Es la misma cuenta, escrita en código rápido.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep02_conv2d.py
__file__ = 'snippets/ep02_conv2d.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn.functional as F

torch.manual_seed(0)

In [9]:
x = torch.randn(1, 1, 8, 8)          # (batch, canales, alto, ancho)
w = torch.randn(1, 1, 3, 3)          # (salidas, entradas, kh, kw)

a_mano = torch.zeros(6, 6)
for i in range(6):
    for j in range(6):
        a_mano[i, j] = (x[0, 0, i:i+3, j:j+3] * w[0, 0]).sum()

con_torch = F.conv2d(x, w)           # el mismo deslizamiento, en C
print("forma:", tuple(con_torch.shape))
print(f"diferencia máxima: {(con_torch[0, 0] - a_mano).abs().max():.1e}")
capa = torch.nn.Conv2d(1, 1, kernel_size=3)
print("weight:", tuple(capa.weight.shape), " bias:", tuple(capa.bias.shape))

forma: (1, 1, 6, 6)
diferencia máxima: 1.4e-06
weight: (1, 1, 3, 3)  bias: (1,)


## Parte 3 · Padding y stride

La salida de la foto quedó un píxel más chica por lado. Hay dos perillas para controlar el tamaño.

### Padding: un marco de ceros

La primera perilla es el padding: agregar un marco alrededor de la imagen antes de deslizar. Acá, la imagen de cinco por cinco con un marco de ceros de un píxel. Ahora mide siete por siete, y el kernel de tres entra en cinco posiciones por lado: la salida vuelve a medir cinco, igual que la entrada. Los píxeles del borde ahora también quedan en el centro de alguna ventana. La segunda perilla es el stride: de a cuántos píxeles salta el kernel. Con stride dos, salta de a dos y la salida queda a la mitad.

### El tamaño de salida

$$
o = \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1
$$
<small>n entrada, k kernel, p padding, s stride</small>

$$
n=32,\ k=3,\ p=1,\ s=2:\ \ \left\lfloor \tfrac{31}{2} \right\rfloor + 1 = 16
$$
<small>un ejemplo con números</small>

$$
p = \tfrac{k-1}{2},\ s = 1 \;\Rightarrow\; o = n
$$
<small>padding «same»: el tamaño no cambia</small>

Todo junto en una fórmula. n es el lado de la entrada, k el del kernel, p el padding y s el stride, todos enteros. El lado de salida o es n más dos p menos k, dividido s, redondeado hacia abajo, que es lo que dibujan esos corchetes sin tapa, más uno. Un ejemplo: treinta y dos de entrada, kernel de tres, padding uno, stride dos. En el dibujo, la suma de arriba: treinta y dos más dos menos tres, treinta y uno. Dividido dos da quince y medio, el piso es quince, más uno: dieciséis. La mitad. Y el caso más usado: con stride uno y padding igual a k menos uno sobre dos, la salida mide lo mismo que la entrada. A eso se le dice padding same.

### La fórmula contra PyTorch

Comprobémoslo. La fórmula como función de Python: la doble barra es la división entera, que ya redondea para abajo. Y para cuatro combinaciones de padding y stride, comparamos con la forma real de F punto conv2d sobre una imagen de treinta y dos. También existe el padding same como palabra. Coinciden las cuatro: treinta sin padding, treinta y dos con padding uno, dieciséis con stride dos, y quince con stride dos sin padding, donde el piso se come una posición.

In [10]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep02_tamanos.py
__file__ = 'snippets/ep02_tamanos.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn.functional as F

In [11]:
x = torch.randn(1, 1, 32, 32)
w = torch.randn(1, 1, 3, 3)

def tam(n, k, p, s):
    return (n + 2 * p - k) // s + 1

for p, s in [(0, 1), (1, 1), (1, 2), (0, 2)]:
    y = F.conv2d(x, w, padding=p, stride=s)
    print(f"p={p} s={s}: fórmula {tam(32, 3, p, s):2d}"
          f"  torch {y.shape[-1]:2d}")
y = F.conv2d(x, w, padding="same")
print("padding='same':", tuple(y.shape[-2:]))

p=0 s=1: fórmula 30  torch 30
p=1 s=1: fórmula 32  torch 32
p=1 s=2: fórmula 16  torch 16
p=0 s=2: fórmula 15  torch 15
padding='same': (32, 32)


> **Dumoulin & Visin, 2016 — A guide to convolution arithmetic for deep learning** · [arXiv:1603.07285](https://arxiv.org/abs/1603.07285)
>
> **o = ⌊(i + 2p − k)/s⌋ + 1** · «Relationship 6. For any i, k, p and s»: la fórmula general del tamaño de salida, con i como lado de la entrada.

Si querés ver esto con dibujos de cada caso, la referencia es la guía de aritmética de convoluciones de Dumoulin y Visin. Su relación número seis es exactamente nuestra fórmula, con i en vez de n para la entrada. La guía también cubre la convolución transpuesta y la dilatada, que acá no vamos a usar.

## Parte 4 · Lo que PyTorch llama convolución

Un detalle que confunde a cualquiera que venga de procesamiento de señales.

### Correlación contra convolución

Usamos scipy, que tiene las dos operaciones con su nombre matemático, y un kernel asimétrico a propósito, para que se note si se da vuelta. La correlación cruzada: deslizar y sumar productos, lo que hicimos todo el episodio. La convolución de los libros de señales. Y la convolución con el kernel girado ciento ochenta grados. F punto conv2d coincide con la correlación, salvo redondeo en la séptima cifra; difiere en más de nueve con la convolución, y vuelve a coincidir si giramos el kernel.

In [12]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep02_correlacion.py
__file__ = 'snippets/ep02_correlacion.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn.functional as F
from scipy.signal import convolve2d, correlate2d

torch.manual_seed(0)

In [13]:
x = torch.randn(6, 6)
w = torch.tensor([[1., 2., 0.],
                  [0., 0., 0.],
                  [0., 0., -1.]])        # asimétrico a propósito
t = F.conv2d(x[None, None], w[None, None])[0, 0].numpy()
xn, wn = x.numpy(), w.numpy()
corr = correlate2d(xn, wn, mode="valid")
conv = convolve2d(xn, wn, mode="valid")
girada = convolve2d(xn, wn[::-1, ::-1], mode="valid")
print(f"torch vs correlación:  {abs(t - corr).max():.1e}")
print(f"torch vs convolución:  {abs(t - conv).max():.2f}")
print(f"torch vs conv. girada: {abs(t - girada).max():.1e}")

torch vs correlación:  4.8e-07
torch vs convolución:  9.27
torch vs conv. girada: 9.5e-07


### La única diferencia: el sentido

$$
(x \star w)_{ij} = \sum_{a,b} x_{i+a,\,j+b}\; w_{ab}
$$
<small>correlación cruzada: lo que hace F.conv2d</small>

$$
(x * w)_{ij} = \sum_{a,b} x_{i-a,\,j-b}\; w_{ab}
$$
<small>convolución: recorre el kernel al revés</small>

$$
x * w = x \star \mathrm{rot}_{180}(w)
$$
<small>girar el kernel convierte una en la otra</small>

En fórmulas. La estrella es la correlación cruzada: suma x en i más a, j más b, con el kernel tal cual; en el dibujo, leído en orden. El asterisco es la convolución de verdad: usa i menos a, j menos b, y en el dibujo el kernel se lee de atrás para adelante. Así que convolucionar es correlacionar con el kernel girado ciento ochenta grados. En una red, el kernel se aprende: si hiciera falta girarlo, el entrenamiento aprende la versión girada. Por eso se dice convolución y da igual.

### Por qué esta operación sirve para ver

- Localidad: cada salida mira solo 3 × 3 vecinos
- Pesos compartidos: 9 números para toda la foto
- Kernels aprendidos: la red elige qué detectar
- Si la imagen se mueve, la salida se mueve igual

Cerremos con por qué esta operación es la base de la visión por computadora. Localidad: cada número de salida depende solo de sus vecinos, que es justo lo que a una red densa le faltaba. Pesos compartidos: los mismos nueve números sirven en toda la foto; detectar un borde arriba o abajo cuesta lo mismo. En una red, esos kernels no los diseña nadie: se aprenden con gradiente, y la red elige qué detectar. Y si la imagen se desplaza, la salida se desplaza igual. Eso lo vamos a medir en el próximo episodio.

## Resumiendo

- Convolución = kernel que se desliza + suma de productos
- Sobel: 9 números fijos encuentran el contorno de la flor
- F.conv2d = el loop; pesos (salidas, entradas, k, k)
- Tamaño: o = ⌊(n + 2p − k)/s⌋ + 1
- PyTorch hace correlación cruzada: da igual si w se aprende
- Próximo: la convolución como multiplicación de matrices